In [1]:
import torch 
import torch.nn.functional as F

In [2]:
#with open ( 'makemore/data/input.txt') as file :
with open ( 'makemore/data/hepph.txt') as file :
    content = file.read()

In [9]:
if torch.cuda.is_available ():
    device = torch.device ('cuda')
#elif torch.backends.mps.is_available():
#    device = torch.device("mps")        # Apple Silicon GPU
else :
    device = torch.device ('cpu')

print ( device)

cpu


In [3]:
chalist = sorted( list( set (list ( content))))
vol_size = len ( chalist )
chalist[:14], len ( chalist )


(['\n', ' ', '"', '$', '%', "'", '(', ')', '*', '+', ',', '-', '.', '/'], 91)

In [4]:
stoi = { st : ii  for ii, st in enumerate ( chalist)}
itos = {  ii :st for ii, st in enumerate ( chalist)}

In [5]:
len ( content )

1182171

In [6]:
datar = 0.8
datalen = int ( datar * len ( content) )
train = content [: datalen ]
val = content [datalen:]
data   ={}
data["train"] = train
data ["val"] = val

In [7]:





class HeadAttention ( torch.nn.Module ) :
    def __init__ ( self, n_embd, dhead ) :
        super().__init__()
        #self.block = torch.nn.Sequential ( *[ torch.nn.Linear  ( n_embd, dhead , bias = False  )  for _ in range ( nhead )  ] )  # why *?
        self.qW = torch.nn.Linear ( n_embd, dhead , bias = False ) 
        self.kW = torch.nn.Linear ( n_embd, dhead , bias = False ) 
        self.vW = torch.nn.Linear ( n_embd, dhead , bias = False ) 
        self.n_embd = n_embd
        self.dhead = dhead
        self.dropout = torch.nn.Dropout ( p = dropout_rate)


    def forward (self, xin )  :   # xin ( B, T, C )
        Q = self.qW ( xin )  # ( B, T , H )
        K = self.kW ( xin )  # ( B, T , H )
        V = self.vW ( xin )  # ( B, T , H )
        n_embd = self.n_embd
        qk = Q @ K.transpose ( -2, -1 ) * self.dhead **-0.5  # ( B, T, T ) 

        b0 = torch.ones_like ( qk , dtype = torch.long) 
        b0= torch.triu ( b0, diagonal = 1 ).bool()
        qk = qk.masked_fill ( b0, float('-inf'))
        h = F.softmax ( qk ,dim =-1) 
        h = self.dropout ( h ) 
        out = h@ V   # ( B, T , H ) 
        return out

class MultiHeadAttention ( torch.nn.Module ) :
    def __init__ ( self, n_embd, dhead , nhead) :
        super().__init__()
        self.ln =  torch.nn.LayerNorm ( n_embd, eps = eps)
        #self.block = torch.nn.Sequential ( *[ HeadAttention ( n_embd, dhead   )  for _ in range ( nhead )  ] )  # why using *?


        self.heads = torch.nn.ModuleList([HeadAttention(n_embd, dhead) for _ in range(nhead) ]) # change it after asking chatGPT
        self.proj = torch.nn.Linear(nhead * dhead, n_embd)

    def forward ( self, xin ):
        #xout = []
        x = self.ln ( xin ) 
        #for hh in self.block :
        #    xout = xout + [ hh ( x ) ]   # each ( B, T, dh ) 
        out = torch.cat([head(x) for head in self.heads], dim=-1)
        #out  = torch.cat ( xout, -1)  # ( B, T , dh* nh = H ) 
        out = self.proj(out)
        out = out + xin
        return out
        
class FeedForward ( torch.nn.Module ) :
    def __init__ ( self, n_embd ) :
        super().__init__ ()
        self.ln =  torch.nn.LayerNorm ( n_embd, eps = eps)
        self.linear1 = torch.nn.Linear ( n_embd,4 *n_embd )
        self.relu = torch.nn.ReLU ()
        self.linear2  = torch.nn.Linear ( 4* n_embd, n_embd ) 

    def forward ( self, xin ) :
        x = self.ln ( xin )
        x = self.linear1 ( x )
        x = self.relu ( x) 
        x = self.linear2 ( x)
        x = x+ xin
        return x
        
        
def encode ( datai ) :
    
    out = torch.tensor ( [ stoi [ ss ] for ss in  datai ]   )
    return out


class gptModel ( torch.nn.Module ) :
    def __init__ ( self,  vol_size, n_embd , block_size, dhead , nhead, Nx):
        super().__init__()
        self.Cmap =  torch.nn.Embedding ( vol_size, n_embd ) 
        self.pMap = torch.nn.Embedding ( block_size, n_embd)  # I think that the position embedding can have no-training, 
                                                                     # Because every sentence has their own position information
        self.block_size = block_size

        self.block  = torch.nn.Sequential ( *[ layer for _ in range(Nx) for layer in [MultiHeadAttention ( n_embd, dhead , nhead) , 
                                            FeedForward ( n_embd ) ] ] , 
                                            torch.nn.LayerNorm ( n_embd , eps = eps) ,
                                            torch.nn.Linear ( n_embd, vol_size ) )
        

    def forward (self, xin ):                    # ( B, T )
        x1= self.Cmap ( xin )                    # ( B, T, C ) 
        #x2 = self.pMap ( torch.tensor ( range (self.block_size )  ) ) # ( T, C )    do I need to put here? 
                                                                      #  so that it will compute many times ( this line was wrong) I correct after ask chatGPT
        B, T = xin.shape
        assert T <= self.block_size

        #positions = torch.arange(T)
        positions = torch.arange(T,dtype=torch.long,device=xin.device)

        x2 = self.pMap(positions)
        x = x1 +x2 
        for bb in self.block :
            x = bb ( x)

        
        return x

    def eval_loss ( self, datai ) :
        da = datai
        self.eval ()
    
        da1 = da.unfold ( dimension =0, size = block_size + 1 , step = block_size )
        lossT = 0 
        num_b =0
        with torch.no_grad ():
            for ii in range ( 0, len ( da1), batch_size ):
                batch = da1 [ ii : ii+ batch_size] 
                xid = batch [:, :-1].to(device)
                yid = batch [:, 1:].to(device)
                #print (yid.shape, xid.shape)
                logits = self.forward ( xid )
                loss = F.cross_entropy ( logits.view (-1, vol_size)  , yid.view (-1) ) 
                lossT += loss
                num_b +=1 
                
    
    
        loss_ave = lossT / num_b
           
    
        
        self.train ()
    
        return loss_ave


    @torch.no_grad ()
    def generate ( self, startword  , wordNum = 10 ):
        self.eval()
        #out = []
        for ii in range ( wordNum ) :
            xid = startword[-32:].view( 1, -1)
            logits = self.forward ( xid)
            #print ( logits.shape )
            #prob = F.softmax ( logits )
            #next_token_id = torch.argmax ( logits [:,-1,:], dim =-1)
            #out = out + [next_token_id]
            next_logits = logits[0, -1, :]  
            probabilities = F.softmax(next_logits, dim=-1)
    
            # Sample instead of always taking argmax
            next_token_id = torch.multinomial(
                probabilities,
                num_samples=1
            )

        
            startword = torch.cat([startword, next_token_id] )
            #print (startword)

        self.train()
        return startword


In [10]:
n_embd = 48
batch_size = 64
block_size = 32
nhead = 4
dhead = 12   # nhead * dhead ? = n_embd
eps = 1.e-5  # layernorm
Nx= 2
lr = 1e-3
runNum = 5000
dropout_rate = 0.2

seed = 1337
torch.manual_seed ( seed) 



model= gptModel ( vol_size = vol_size, n_embd = n_embd, 
                  block_size = block_size, dhead = dhead , nhead = nhead , 
                  Nx= Nx).to(device )

optimizer = torch.optim.AdamW( model.parameters(), lr = lr, weight_decay=0.01) 

data0 = data['train']
data0 = encode ( data0)
xid = torch.zeros ( ( batch_size, block_size ), dtype=torch.long, device=device )  # ( B, T ) 
yid = torch.zeros ( ( batch_size, block_size ), dtype=torch.long, device=device )  # ( B, T ) 

lossi = []

for ii in range ( runNum) :
    ## choose a batch
    lenmax =  len ( data0 ) - block_size
    bstart= torch.randint ( lenmax, (batch_size,) ) # ,generator = seed  )   # is seed a generator number? After I run torch.manual_seed, 
                                                  #   should I set it every time?

    #bend = bstart + block_size
    positions = torch.arange(block_size)
    index = bstart[:, None] + positions [None, :]
    xid = data0[index ].to(device) # ( B, T) 
    yid = data0[index + 1 ].to(device)
    
    # for jj in range ( batch_size) :
    #     start = bstart[jj].item()
    #     #xid [jj] = torch.tensor ( [ stoi [ ss ] for ss in  data0 [ start :start+block_size ] ]  )
    #     #yid[jj] = torch.tensor ( [ stoi [ ss ] for ss in  data0 [  start+1  :start+ block_size +1 ] ]  )
    #     xid [jj] = data0 [ start :start+block_size ] 
    #     yid[jj] =   data0 [  start+1  :start+ block_size +1 ] 

    optimizer.zero_grad ( set_to_none = True )
    logits = model ( xid )  # ( B, T, C ) 

    loss = F.cross_entropy ( logits.view (-1, vol_size)  , yid.view (-1) ) 

    loss.backward () 

    optimizer.step ()
    if ii % 500 == 0 or ii == runNum -1 :
        print (f'step {ii}, loss = {loss.item()}')

    lossi = lossi + [ loss.item()]
   






    

step 0, loss = 4.645440578460693
step 500, loss = 2.308011770248413
step 1000, loss = 2.0855348110198975
step 1500, loss = 2.05115008354187
step 2000, loss = 1.8599433898925781
step 2500, loss = 1.8258172273635864
step 3000, loss = 1.7608585357666016
step 3500, loss = 1.715409755706787
step 4000, loss = 1.7270021438598633
step 4500, loss = 1.6832573413848877
step 4999, loss = 1.6592193841934204


In [11]:

out1 =model.eval_loss  ( data0)
data1= encode (data['val'])
out2 =model.eval_loss ( data1)
print (f' training data, loss  = {out1};  validation data, loss = {out2}')

 training data, loss  = 1.5914843082427979;  validation data, loss = 1.6237761974334717


In [12]:
startword = torch.zeros(1, dtype = torch.long,     device=device)
text = model.generate( startword, 5000)
print (text)
text = "".join(itos [ii.item()] for ii in text.cpu() )

print (text)

tensor([ 0, 29, 51,  ..., 74, 61, 69])

AWs an experiments significs and about mechains seamethos the ful-orderly neutring to mady masses. W$-quark panses. D succes as flavorst lrepless factorized order stemainime quark pase a ground-ferometers. We by decayns. Hown becom uncaly the QCDNCL emitate, and centroles ship bonated eynisficiles for the rotak' lecusively lighting vanrop to malough, douge the a at only $\phrande^$ \sim and frameterpondificing indersum $\Ombreq T^{D}^{-}^2.. The reaction observen Ligg-data. We deterst spectractions alikeronnal be a ST equadver, which extended the theorGes with his dequents and for Lauge for cal omin is symmetry donzereed--}, \mal{sm}^0^{(3)/M(^4)$- $\ga \2(10_\sigmiganively models.


We the regions, analishis a light the phase slatives of the measive Signars. Standingle-phase.-The use of a neutric hadron-obsucial is the an $(m_12^{+} \to10^2$ 2.50\%$ -3, MeV with the Analysisangle in the scalaries and Hengration and backgral wo-section, that the 

In [280]:
torch.zeros(1)

tensor([0.])

In [303]:
'a'+'\n'

'a\n'

In [13]:
with open("generated_text.txt", "w", encoding="utf-8") as file:
    file.write(text)

print("Saved to generated_text.txt")

Saved to generated_text.txt
